# Backtest EMACross

Run the EMACross strategy on a simulated exchange venue.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + EMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweep — fast/slow grid with heatmap

**Prerequisites:** Run `scripts/fetch_binance_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_sweep
from src.core import bar_type_str, with_venue_config
from src.strategies.ma_cross import MACross, MACrossConfig

from charts import plot_ma_cross, plot_equity_curve, print_summary_stats, plot_pnl_heatmap, generate_backtest_html
from utils import make_instrument_id, save_notebook, save_notebook_html, save_tearsheet

# ── Shared config ────────────────────────────────────────────────
EXCHANGE         = "BINANCE"
ASSET            = "BTC"
BAR_INTERVAL     = "1d"
STARTING_CAPITAL = 10_000
TRADE_SIZE       = 1000              # $1,000 USD per trade
LEVERAGE         = 20                # Backtesting leverage (margin_init = 1/20 = 5%)
SAVE_TEARSHEET   = False

# MA params
MA_TYPE  = "EMA"
FAST_MA  = 20
SLOW_MA  = 50

# Sweep grids
FAST_PERIODS = sorted(set([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100] + [FAST_MA]))
SLOW_PERIODS = sorted(set([10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200] + [SLOW_MA]))

# Standard
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, EXCHANGE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"
RESULT_NAME  = f"MACross-{MA_TYPE}_{INSTRUMENT_ID}_{BAR_INTERVAL}_f{FAST_MA}_s{SLOW_MA}"

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

# Override margins for backtesting (catalog instruments have raw/live margins)
instrument = with_venue_config(instrument, LEVERAGE)

# Settlement currency for PnL queries (USDC for HL, USDT for Binance)
CURRENCY = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Currency   : {CURRENCY}")
print(f"Leverage   : {LEVERAGE}x (margin_init={instrument.margin_init}, margin_maint={instrument.margin_maint})")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")
print(f"Maker fee  : {instrument.maker_fee}")
print(f"Taker fee  : {instrument.taker_fee}")

# Optional: filter bars by date range (comment out for full dataset)
DATE_FROM = pd.Timestamp("2025-01-01", tz="UTC")
DATE_TO   = pd.Timestamp("2025-04-01", tz="UTC")
filtered_bars = [b for b in bars if DATE_FROM.value <= b.ts_event <= DATE_TO.value]
print(f"Filtered: {len(filtered_bars)} bars ({DATE_FROM:%Y-%m-%d} → {DATE_TO:%Y-%m-%d})")


In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add MACross strategy + run ────────────────────────────
config = MACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_notional=TRADE_NOTIONAL,
    ma_type=MA_TYPE,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
)
strategy = MACross(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA}) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

if SAVE_TEARSHEET:
    save_tearsheet(html, RESULT_NAME)

In [ ]:
# ── Cell 8: Price chart with MA overlay + trade markers ───────────

fig = plot_ma_cross(
    bars, fills_report,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    ma_type=MA_TYPE,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ──────────────────────────────────────────
plot_equity_curve(analyzer, account_report, f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA})  {INSTRUMENT_ID} {BAR_INTERVAL}")

In [ ]:
# ── Cell 10: Summary statistics ────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

In [ ]:
# ── Cell 11: Parameter sweep ──────────────────────────────────────
#
# Grid search over fast/slow MA period combinations.

def ma_factory(eng, params):
    cfg = MACrossConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_notional=TRADE_NOTIONAL,
        ma_type=MA_TYPE,
        fast_period=params["fast"],
        slow_period=params["slow"],
    )
    eng.add_strategy(MACross(cfg))

combos = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=ma_factory,
    strategy_name=f"MACross-{MA_TYPE}",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

In [ ]:
# ── Cell 12: PnL heatmap ──────────────────────────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label=f"Slow {MA_TYPE} Period", col_label=f"Fast {MA_TYPE} Period",
    title=f"Total PnL ({CURRENCY}) by {MA_TYPE} Parameters",
)

In [ ]:
# -- Cell 13: TradingView Interactive Backtest ------------------

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    ma_type=MA_TYPE,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=RESULT_NAME,
    open_browser=True,
)

In [ ]:
# ── Cell 14: Save notebook snapshot ────────────────────────────────────────
#
# Save the notebook (Ctrl+S) before running this cell.

#save_notebook("backtest_ema_cross.ipynb", RESULT_NAME)
#save_notebook_html("backtest_ema_cross.ipynb", RESULT_NAME)

In [ ]:
# ── Cell 15: Cleanup ──────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")
